
## 小專題：今天臺灣哪裡空氣比較差？

- 資料集名稱：[空氣品質指標(AQI)](https://data.gov.tw/dataset/40448)

請使用 Pandas 分析最新的空氣品質資料，回答：

這份資料有幾筆資料、幾個欄位？
- 是否存在缺失值？
- 是否存在重複資料？
- 找出 AQI 大於等於 70 的測站。
- 找出高雄市的所有測站。
- 找出 PM2.5 大於 18 的測站。
- 新增一個「是否需要注意」欄位。
- 整理出只包含「縣市、測站、AQI、PM2.5、狀態」的結果。
- 將分析結果輸出成 CSV。

### 下載 AQI CSV

In [4]:
import requests
import pandas as pd

### 設定 CSV 資料網址
# url = "這裡貼上政府資料開放平臺的 CSV 網址"
url = "https://data.moenv.gov.tw/api/v2/aqx_p_432?api_key=af57253c-e838-46da-a1f5-12b43afd75f3&limit=1000&sort=ImportDate%20desc&format=CSV"

headers = {
    "User-Agent": "Mozilla/5.0"
}


### 發送 HTTP 請求
response = requests.get(
    url,
    headers=headers,
    timeout=30
) # 向政府開放資料網站發送 GET 請求

response.raise_for_status() # 如果發生 4xx、5xx 錯誤，直接拋出例外

print("狀態碼：", response.status_code)


### 儲存 CSV
with open("../pandas_datasets/空氣品質指標.csv", "wb") as file:
    file.write(response.content) # 將伺服器回傳的 CSV 原始資料寫入檔案

print("CSV 下載完成")


### 使用 Pandas 讀取 CSV
df = pd.read_csv("../pandas_datasets/空氣品質指標.csv")
print("\n資料讀取完成")

狀態碼： 200
CSV 下載完成

資料讀取完成


In [5]:
### 查看資料
print("\n前 5 筆資料：")
df.head()


前 5 筆資料：


,sitename,county,aqi,pollutant,status,so2,co,o3,o3_8hr,pm10,...,wind_speed,wind_direc,publishtime,co_8hr,pm2.5_avg,pm10_avg,so2_avg,longitude,latitude,siteid
0,基隆,基隆市,42,NaN,良好,1.5,0.27,37,23,26.0,...,0.6,85,2026/09/09 23:00:00,0.3,8.7,25.0,1.0,121.760056,25.129168,1
1,汐止,新北市,48,NaN,良好,1.0,0.32,13,22,36.0,...,0.4,65,2026/09/09 23:00:00,0.3,10.9,25.0,1.0,121.640810,25.066240,2
2,新店,新北市,38,NaN,良好,0.7,0.28,17,24,20.0,...,1.1,192,2026/09/09 23:00:00,0.3,9.5,17.0,0.0,121.537780,24.977222,4
3,土城,新北市,37,NaN,良好,1.3,0.27,32,33,22.0,...,0.8,163,2026/09/09 23:00:00,0.2,8.8,22.0,NaN,121.451860,24.982529,5
4,板橋,新北市,42,NaN,良好,1.3,0.44,22,28,20.0,...,0.3,99,2026/09/09 23:00:00,0.4,10.3,18.0,1.0,121.458664,25.012972,6


In [11]:
### 查看資料大小
print("\n資料大小：", df.shape)



資料大小： (84, 24)


In [9]:
### 查看欄位
print("\n欄位名稱：")
df.columns


欄位名稱：


Index(['sitename', 'county', 'aqi', 'pollutant', 'status', 'so2', 'co', 'o3',
       'o3_8hr', 'pm10', 'pm2.5', 'no2', 'nox', 'no', 'wind_speed',
       'wind_direc', 'publishtime', 'co_8hr', 'pm2.5_avg', 'pm10_avg',
       'so2_avg', 'longitude', 'latitude', 'siteid'],
      dtype='str')

In [12]:
### 檢查缺失值
print("\n每個欄位的缺失值數量：")
df.isna().sum()


每個欄位的缺失值數量：


sitename        0
county          0
aqi             0
pollutant      45
status          0
so2             1
co              0
o3              0
o3_8hr          0
pm10            1
pm2.5           0
no2             0
nox             0
no              0
wind_speed      2
wind_direc      2
publishtime     0
co_8hr          0
pm2.5_avg       0
pm10_avg        1
so2_avg         1
longitude       0
latitude        0
siteid          0
dtype: int64

### 檢查重複資料

In [13]:
print("\n重複資料數量：")
df.duplicated().sum()


重複資料數量：


np.int64(0)

### 移除完全重複的資料

In [14]:
df = df.drop_duplicates()

### 將文字轉換成數字

In [15]:
df.dtypes # 先查看所有欄位資料型態

sitename           str
county             str
aqi              int64
pollutant          str
status             str
so2            float64
co             float64
o3               int64
o3_8hr           int64
pm10           float64
pm2.5            int64
no2              int64
nox            float64
no             float64
wind_speed         str
wind_direc         str
publishtime        str
co_8hr         float64
pm2.5_avg      float64
pm10_avg       float64
so2_avg        float64
longitude      float64
latitude       float64
siteid           int64
dtype: object

In [ ]:
### 將 AQI 轉換成數字
df["aqi"] = pd.to_numeric(
    df["aqi"],
    errors="coerce" # 轉換失敗不要報錯，把有問題的資料改成缺失值(NaN)。
)


### 將 PM2.5 轉換成數字
df["pm2.5"] = pd.to_numeric(
    df["pm2.5"],
    errors="coerce" # 轉換失敗不要報錯，把有問題的資料改成缺失值(NaN)。
)

### 找出 AQI 大於等於 70 的測站

In [24]:
print("\nAQI 大於等於 70 的測站：")
high_aqi = df.loc[
    df["aqi"] >= 70,
    [
        "county",
        "sitename",
        "aqi",
        "pm2.5",
        "pollutant",
        "status"
    ]
]
high_aqi


AQI 大於等於 70 的測站：


,county,sitename,aqi,pm2.5,pollutant,status
50,高雄市,林園,71,15,臭氧八小時,普通
70,連江縣,馬祖,80,11,臭氧八小時,普通
80,南投縣,南投（鹿谷）,97,18,臭氧八小時,普通


### 找出高雄市測站

In [25]:
kaohsiung = df.loc[
    df["county"] == "高雄市",
    [
        "county",
        "sitename",
        "aqi",
        "pm2.5",
        "status"
    ]
]
kaohsiung

,county,sitename,aqi,pm2.5,status
45,高雄市,美濃,50,9,良好
46,高雄市,橋頭,57,21,普通
47,高雄市,仁武,54,19,普通
48,高雄市,鳳山,64,21,普通
49,高雄市,大寮,61,18,普通
50,高雄市,林園,71,15,普通
51,高雄市,楠梓,51,15,普通
52,高雄市,左營,58,17,普通
53,高雄市,前金,54,15,普通
54,高雄市,前鎮,54,17,普通


### 找出 PM2.5 大於 18 的測站

In [32]:
high_pm25 = df.loc[
    df["pm2.5"] > 18,
    [
        "county",
        "sitename",
        "aqi",
        "pm2.5",
        "status"
    ]
]
high_pm25

,county,sitename,aqi,pm2.5,status
46,高雄市,橋頭,57,21,普通
47,高雄市,仁武,54,19,普通
48,高雄市,鳳山,64,21,普通
68,高雄市,復興,51,20,普通
69,南投縣,埔里,69,20,普通


### 新增「是否需要注意」欄位

In [33]:
df["是否需要注意"] = df["aqi"] >= 70
df

,sitename,county,aqi,pollutant,status,so2,co,o3,o3_8hr,pm10,...,wind_direc,publishtime,co_8hr,pm2.5_avg,pm10_avg,so2_avg,longitude,latitude,siteid,是否需要注意
0,基隆,基隆市,42,NaN,良好,1.5,0.27,37,23,26.0,...,85,2026/09/09 23:00:00,0.3,8.7,25.0,1.0,121.760056,25.129168,1,False
1,汐止,新北市,48,NaN,良好,1.0,0.32,13,22,36.0,...,65,2026/09/09 23:00:00,0.3,10.9,25.0,1.0,121.640810,25.066240,2,False
2,新店,新北市,38,NaN,良好,0.7,0.28,17,24,20.0,...,192,2026/09/09 23:00:00,0.3,9.5,17.0,0.0,121.537780,24.977222,4,False
3,土城,新北市,37,NaN,良好,1.3,0.27,32,33,22.0,...,163,2026/09/09 23:00:00,0.2,8.8,22.0,NaN,121.451860,24.982529,5,False
4,板橋,新北市,42,NaN,良好,1.3,0.44,22,28,20.0,...,99,2026/09/09 23:00:00,0.4,10.3,18.0,1.0,121.458664,25.012972,6,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,林森,臺南市,57,懸浮微粒,普通,1.2,0.44,36,48,34.0,...,79,2026/09/09 23:00:00,0.4,13.0,36.0,1.0,120.222160,22.985226,140,False
80,南投（鹿谷）,南投縣,97,臭氧八小時,普通,2.2,0.21,66,69,20.0,...,255,2026/09/09 23:00:00,0.2,14.6,26.0,2.0,120.794480,23.718676,203,True
81,屏東（琉球）,屏東縣,51,臭氧八小時,普通,0.5,0.16,58,55,29.0,...,162,2026/09/09 23:00:00,0.1,10.1,23.0,1.0,120.377220,22.352220,204,False
82,新北（樹林）,新北市,47,NaN,良好,0.1,0.32,37,30,19.0,...,112,2026/09/09 23:00:00,0.3,11.6,22.0,0.0,121.383530,24.949028,311,False


### 整理出只包含「縣市、測站、AQI、PM2.5、狀態」的結果


In [34]:
### 只保留分析需要的欄位
result = df[
    [
        "county",
        "sitename",
        "aqi",
        "pm2.5",
        "status",
        "是否需要注意"
    ]
]

### 修改中文欄位名稱
result = result.rename(
    columns={
        "county": "縣市",
        "sitename": "測站",
        "aqi": "AQI",
        "pm2.5": "PM2.5",
        "status": "空氣品質狀態"
    }
)
result

,縣市,測站,AQI,PM2.5,空氣品質狀態,是否需要注意
0,基隆市,基隆,42,6,良好,False
1,新北市,汐止,48,10,良好,False
2,新北市,新店,38,10,良好,False
3,新北市,土城,37,8,良好,False
4,新北市,板橋,42,11,良好,False
...,...,...,...,...,...,...
79,臺南市,林森,57,12,普通,False
80,南投縣,南投（鹿谷）,97,18,普通,True
81,屏東縣,屏東（琉球）,51,10,普通,False
82,新北市,新北（樹林）,47,10,良好,False


### 將分析結果輸出成 CSV

In [35]:
### 輸出 CSV
result.to_csv(
    "../pandas_output/臺灣空氣品質分析.csv",
    index=False,
    encoding="utf-8-sig"
)


### 輸出高 AQI 測站
high_aqi.to_csv(
    "../pandas_output/AQI超過70測站.csv",
    index=False,
    encoding="utf-8-sig"
)